# Correlation Analysis - L'Oreal India Example

This notebook demonstrates correlation calculation and visualization using L'Oreal India sales and marketing data.

**Key Learning**: Correlation measures how two variables move together, but does NOT prove causation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Generate L'Oreal India Monthly Data

We simulate 24 months of data with:
- Ad spend (in Lakhs)
- Season factor (Diwali = high, off-season = low)
- Sales (in Crores)

**Important**: Season affects BOTH ad spend AND sales (confounder).

In [ ]:
def generate_loreal_data(n_months=24, seed=42):
    """Generate synthetic L'Oreal India monthly data."""
    np.random.seed(seed)
    
    data = {
        'month': pd.date_range('2024-01', periods=n_months, freq='M'),
        'ad_spend_lakhs': np.random.normal(60, 15, n_months).clip(20, 100),
        'season_factor': np.tile([0.8, 0.9, 1.0, 1.1, 1.0, 0.9, 
                                   0.85, 0.9, 1.0, 1.3, 1.4, 1.2], 2)[:n_months],
    }
    
    # Sales influenced by both ad spend AND season (confounder)
    base_sales = 100
    data['sales_crores'] = (
        base_sales 
        + 0.3 * data['ad_spend_lakhs']  # True ad effect
        + 50 * data['season_factor']     # Season effect
        + np.random.normal(0, 5, n_months)  # Noise
    )
    
    return pd.DataFrame(data)

# Generate data
df = generate_loreal_data()
print("L'Oreal India - Monthly Data (First 12 months)")
print("=" * 60)
df.head(12)

## 2. Visualize the Data

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Time series - Ad Spend
axes[0, 0].plot(df['month'], df['ad_spend_lakhs'], marker='o', color='blue')
axes[0, 0].set_title('Ad Spend Over Time')
axes[0, 0].set_ylabel('Ad Spend (Lakhs)')
axes[0, 0].tick_params(axis='x', rotation=45)

# Time series - Sales
axes[0, 1].plot(df['month'], df['sales_crores'], marker='o', color='green')
axes[0, 1].set_title('Sales Over Time')
axes[0, 1].set_ylabel('Sales (Crores)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Scatter plot - Ad Spend vs Sales
axes[1, 0].scatter(df['ad_spend_lakhs'], df['sales_crores'], alpha=0.7, s=100)
axes[1, 0].set_xlabel('Ad Spend (Lakhs)')
axes[1, 0].set_ylabel('Sales (Crores)')
axes[1, 0].set_title('Ad Spend vs Sales')

# Add trend line
z = np.polyfit(df['ad_spend_lakhs'], df['sales_crores'], 1)
p = np.poly1d(z)
axes[1, 0].plot(df['ad_spend_lakhs'].sort_values(), 
                p(df['ad_spend_lakhs'].sort_values()), 
                "r--", alpha=0.8, label='Trend')
axes[1, 0].legend()

# Season factor
axes[1, 1].bar(range(len(df)), df['season_factor'], color='orange', alpha=0.7)
axes[1, 1].set_xlabel('Month Index')
axes[1, 1].set_ylabel('Season Factor')
axes[1, 1].set_title('Seasonality (Diwali = Oct-Nov)')
axes[1, 1].axhline(y=1.0, color='red', linestyle='--', label='Baseline')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3. Calculate Correlations

In [ ]:
def calculate_correlation(x, y):
    """Calculate Pearson correlation with p-value."""
    r, p_value = stats.pearsonr(x, y)
    return r, p_value

def interpret_correlation(r):
    """Interpret correlation strength."""
    abs_r = abs(r)
    if abs_r >= 0.7:
        strength = "strong"
    elif abs_r >= 0.4:
        strength = "moderate"
    elif abs_r >= 0.1:
        strength = "weak"
    else:
        strength = "negligible"
    
    direction = "positive" if r > 0 else "negative"
    return f"{strength} {direction}"

# Calculate all correlations
r_ads_sales, p_ads = calculate_correlation(df['ad_spend_lakhs'], df['sales_crores'])
r_season_sales, p_season = calculate_correlation(df['season_factor'], df['sales_crores'])
r_ads_season, p_ads_season = calculate_correlation(df['ad_spend_lakhs'], df['season_factor'])

print("Correlation Results")
print("=" * 60)
print(f"\n1. Ad Spend vs Sales:")
print(f"   r = {r_ads_sales:.3f} ({interpret_correlation(r_ads_sales)})")
print(f"   p-value = {p_ads:.4f}")

print(f"\n2. Season vs Sales:")
print(f"   r = {r_season_sales:.3f} ({interpret_correlation(r_season_sales)})")
print(f"   p-value = {p_season:.4f}")

print(f"\n3. Ad Spend vs Season (confounder check):")
print(f"   r = {r_ads_season:.3f} ({interpret_correlation(r_ads_season)})")
print(f"   p-value = {p_ads_season:.4f}")

## 4. Correlation Matrix Heatmap

In [ ]:
# Calculate correlation matrix
corr_matrix = df[['ad_spend_lakhs', 'season_factor', 'sales_crores']].corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)

# Add labels
labels = ['Ad Spend', 'Season', 'Sales']
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

# Add correlation values
for i in range(len(labels)):
    for j in range(len(labels)):
        text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                       ha='center', va='center', color='black', fontsize=14)

plt.colorbar(im, label='Correlation')
plt.title('Correlation Matrix - L\'Oreal India Data')
plt.tight_layout()
plt.show()

## 5. The Confounding Problem

The DAG (Directed Acyclic Graph) shows the true causal structure:

```
        Season (Confounder)
           /       \
          v         v
     Ad Spend ----> Sales
```

Season affects BOTH:
- Ad Spend (companies spend more during Diwali)
- Sales (customers buy more during Diwali)

This creates **spurious correlation** between Ad Spend and Sales!

In [ ]:
# Demonstrate the confounding effect
print("The Confounding Effect")
print("=" * 60)

# Split by high/low season
high_season = df[df['season_factor'] >= 1.1]
low_season = df[df['season_factor'] < 1.0]

print("\nHigh Season (Diwali period):")
print(f"  Avg Ad Spend: Rs {high_season['ad_spend_lakhs'].mean():.1f} Lakhs")
print(f"  Avg Sales: Rs {high_season['sales_crores'].mean():.1f} Crores")

print("\nLow Season (Off-peak):")
print(f"  Avg Ad Spend: Rs {low_season['ad_spend_lakhs'].mean():.1f} Lakhs")
print(f"  Avg Sales: Rs {low_season['sales_crores'].mean():.1f} Crores")

print("\n" + "=" * 60)
print("INSIGHT:")
print("=" * 60)
print("""
Both Ad Spend AND Sales are higher during Diwali season.
This inflates the observed correlation between them.

The TRUE causal effect of ads is much smaller than
the correlation suggests!

CORRELATION != CAUSATION
""")

## 6. Controlling for the Confounder

To estimate the TRUE effect of Ad Spend, we need to control for Season.

In [ ]:
from sklearn.linear_model import LinearRegression

# Naive regression (ignoring confounder)
X_naive = df[['ad_spend_lakhs']]
y = df['sales_crores']
model_naive = LinearRegression().fit(X_naive, y)

# Adjusted regression (controlling for season)
X_adjusted = df[['ad_spend_lakhs', 'season_factor']]
model_adjusted = LinearRegression().fit(X_adjusted, y)

print("Regression Analysis")
print("=" * 60)

print("\n1. NAIVE Model (ignoring season):")
print(f"   Sales = {model_naive.intercept_:.2f} + {model_naive.coef_[0]:.2f} * Ad_Spend")
print(f"   Effect: Rs 1 Lakh more ad spend -> Rs {model_naive.coef_[0]:.2f} Cr more sales")

print("\n2. ADJUSTED Model (controlling for season):")
print(f"   Sales = {model_adjusted.intercept_:.2f} + {model_adjusted.coef_[0]:.2f} * Ad_Spend + {model_adjusted.coef_[1]:.2f} * Season")
print(f"   Effect: Rs 1 Lakh more ad spend -> Rs {model_adjusted.coef_[0]:.2f} Cr more sales")

print("\n" + "=" * 60)
print("COMPARISON:")
print("=" * 60)
print(f"\nNaive effect estimate:    {model_naive.coef_[0]:.2f} Cr per Lakh")
print(f"Adjusted effect estimate: {model_adjusted.coef_[0]:.2f} Cr per Lakh")
print(f"True effect (from data generation): 0.30 Cr per Lakh")
print(f"\nThe naive estimate is BIASED due to confounding!")

## Key Takeaways

1. **Correlation measures relationship strength** - but not causation
2. **Confounders create spurious correlations** - Season affects both Ad Spend and Sales
3. **Naive analysis overestimates effects** - because it includes confounder's influence
4. **Controlling for confounders** reveals the true causal effect
5. **Always ask**: "What else could explain this relationship?"